# LLM-as-a-Judge with RunPod-Hosted Models

This notebook judges OSQ responses using open-source models hosted on RunPod.

**Architecture:**
- Provision N RunPod pods with GPU
- Each pod hosts the same judge model (e.g., gpt-oss:120b) via Ollama
- Distribute evaluated models across pods
- Each pod judges its assigned models sequentially
- Download results to Phase 5 directory

**Key Features:**
- Pod-level parallelism (no within-pod threading)
- Append-only resumable design
- Phase alignment checks
- Compatible with existing llm-judge pipeline

# Configuration

In [14]:
import os
import time
import json
import stat
import posixpath
import runpod
import paramiko
import requests
from pathlib import Path
from datetime import datetime
from copy import deepcopy
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# ───────────────────────────────────────────────
# RUNPOD CONFIGURATION
# ───────────────────────────────────────────────
RUNPOD_API_KEY = os.environ["RUNPOD_API_KEY"]
runpod.api_key = RUNPOD_API_KEY

IMAGE_NAME = "runpod/pytorch:2.8.0-py3.11-cuda12.8.1-cudnn-devel-ubuntu22.04"
GPU_TYPE = "NVIDIA H200"  # Use H200 for large models like gpt-oss:120b
GPU_COUNT = 1
CONTAINER_DISK_GB = 200

# ───────────────────────────────────────────────
# JUDGE CONFIGURATION
# ───────────────────────────────────────────────
TASK_NAME = "sysengbench-osq"
JUDGE_MODEL = "gpt-oss:120b"  # Ollama model to use for judging
PROMPT_ID = "p1"  # "p1" for scores only, "p2" for scores + justifications
TEMPERATURE = 0.0
MAX_TOKENS = 2000
SAMPLE_N = 0  # 0 = judge ALL samples; else judge first N (for testing)

# ───────────────────────────────────────────────
# PARALLELISM
# ───────────────────────────────────────────────
MAX_CONCURRENT_PODS = 2  # Number of pods to run in parallel

# ───────────────────────────────────────────────
# PATHS
# ───────────────────────────────────────────────
BASE_DIR = Path.cwd()
PHASE4_ROOT = BASE_DIR / "../phase4_inference/output" / TASK_NAME
PHASE5_ROOT = BASE_DIR / f"{TASK_NAME}-llm-judge"
# PHASE5_ROOT = BASE_DIR / f"{TASK_NAME}-llm-judge-TEST" # use this one to dump files in a different dir/ for troubleshooting.
LOG_DIR = BASE_DIR / "runpod_llm_judge_logs"

PHASE5_ROOT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

SSH_KEY_PATH = os.path.expanduser("~/.ssh/id_ed25519")

print(f"PHASE4_ROOT: {PHASE4_ROOT}")
print(f"PHASE5_ROOT: {PHASE5_ROOT}")
print(f"LOG_DIR: {LOG_DIR}")
print(f"RunPod API Key present: {bool(RUNPOD_API_KEY)}")

PHASE4_ROOT: c:\Users\rabel\Desktop\dissertation\src\phase5_llm_as_a_judge\..\phase4_inference\output\sysengbench-osq
PHASE5_ROOT: c:\Users\rabel\Desktop\dissertation\src\phase5_llm_as_a_judge\sysengbench-osq-llm-judge
LOG_DIR: c:\Users\rabel\Desktop\dissertation\src\phase5_llm_as_a_judge\runpod_llm_judge_logs
RunPod API Key present: True


# Judge Prompts

In [15]:
# Prompt 1: Scores only
JUDGE_PROMPT_P1 = """You are an expert systems engineering (SE) educator evaluating student responses to open-ended questions.

EVALUATION SCHEMA (each 0–20, total 100):
1. TECHNICAL ACCURACY — correctness of SE concepts, terminology, and facts
2. CONCEPTUAL UNDERSTANDING — depth of comprehension of underlying SE principles
3. COMPLETENESS — coverage of key elements expected in the answer
4. CLARITY & ORGANIZATION — logical structure and clear communication
5. PROFESSIONAL RELEVANCE — connection to real-world SE practice and standards

QUESTION CONTEXT:
Original Question: {osq_question}
Expected Answer: {expected_answer}
Bloom's Level: {blooms_level}
SE Domain: {se_domain}

STUDENT RESPONSE TO EVALUATE:
{student_response}

INSTRUCTIONS:
Compare the response to the expected answer and criteria.
Return strict JSON in this format:
{{
  "technical_accuracy": {{"score": <0–20>}},
  "conceptual_understanding": {{"score": <0–20>}},
  "completeness": {{"score": <0–20>}},
  "clarity_organization": {{"score": <0–20>}},
  "professional_relevance": {{"score": <0–20>}},
  "overall_score": <0–100>
}}
"""

# Prompt 2: Scores + justifications
JUDGE_PROMPT_P2 = """You are an expert systems engineering (SE) educator evaluating student responses to open-ended questions.

EVALUATION SCHEMA (each 0–20, total 100):
1. TECHNICAL ACCURACY — correctness of SE concepts, terminology, and facts
2. CONCEPTUAL UNDERSTANDING — depth of comprehension of underlying SE principles
3. COMPLETENESS — coverage of key elements expected in the answer
4. CLARITY & ORGANIZATION — logical structure and clear communication
5. PROFESSIONAL RELEVANCE — connection to real-world SE practice and standards

QUESTION CONTEXT:
Original Question: {osq_question}
Expected Answer: {expected_answer}
Bloom's Level: {blooms_level}
SE Domain: {se_domain}

STUDENT RESPONSE TO EVALUATE:
{student_response}

INSTRUCTIONS:
Compare the response to the expected answer and criteria.
Return strict JSON in this format:
{{
  "technical_accuracy": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "conceptual_understanding": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "completeness": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "clarity_organization": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "professional_relevance": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "overall_score": <0–100>,
  "overall_assessment": "<summary>",
  "key_strengths": "<strengths>",
  "improvement_areas": "<areas for improvement>"
}}
"""

# Select prompt based on PROMPT_ID
JUDGE_PROMPT = JUDGE_PROMPT_P1 if PROMPT_ID == "p1" else JUDGE_PROMPT_P2

# Helper Functions

In [16]:
def newest(path_iter):
    """Return the most recently modified file from an iterator."""
    items = sorted([p for p in path_iter], key=lambda p: p.stat().st_mtime, reverse=True)
    return items[0] if items else None

def load_jsonl(path: Path):
    """Load all lines from a JSONL file."""
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def safe_json(s: str):
    """Safely parse JSON, return None on failure."""
    try:
        return json.loads(s)
    except Exception:
        return None

def extract_student_response(sample_row):
    """Extract student response from Phase 4 sample row."""
    resps = sample_row.get("resps")
    if not resps:
        return ""
    first = resps[0]
    if isinstance(first, list) and first:
        first = first[0]
    return first if isinstance(first, str) else ""

def derive_prompt_fields(phase4_row):
    """Extract fields needed for judge prompt from Phase 4 row."""
    doc = (phase4_row.get("doc") or {}) if isinstance(phase4_row.get("doc"), dict) else {}
    osq_question = doc.get("osq_prompt") or doc.get("question") or ""
    expected_answer = doc.get("expected_answer", "")
    blooms_level = doc.get("blooms_level", "N/A")
    se_domain = doc.get("INCOSE Handbook Category", "General Systems Engineering")
    student_resp = extract_student_response(phase4_row)
    return {
        "osq_question": osq_question,
        "expected_answer": expected_answer,
        "student_response": student_resp,
        "blooms_level": blooms_level,
        "se_domain": se_domain,
    }

def triad_missing(fields):
    """Check if required fields are missing."""
    return [
        k for k in ("osq_question", "expected_answer", "student_response")
        if not isinstance(fields.get(k, ""), str) or not fields.get(k, "").strip()
    ]

def ensure_alignment_or_die(existing_row, current_src_row, sample_id):
    """Verify Phase 4 data hasn't changed since judging started."""
    snap = existing_row.get("phase4_row")
    if snap is None:
        raise RuntimeError(
            f"[ALIGNMENT ERROR] sample_id={sample_id} has no 'phase4_row' snapshot."
        )
    if snap != current_src_row:
        raise RuntimeError(
            f"[ALIGNMENT ERROR] sample_id={sample_id} Phase-4 content changed.\n"
            "Refusing to proceed. Freeze Phase-4 or regenerate Phase-5 from scratch."
        )

# RunPod Helper Functions

In [17]:
def run_and_check(ssh, cmd, desc, log_func):
    """Run a remote command and wait for completion."""
    log_func(f"▶ {desc}")
    stdin, stdout, stderr = ssh.exec_command(cmd)
    exit_code = stdout.channel.recv_exit_status()
    out = stdout.read().decode()
    err = stderr.read().decode()
    if out:
        log_func(out)
    if err:
        log_func(f"stderr: {err}")
    if exit_code != 0:
        raise RuntimeError(f"{desc} failed with exit code {exit_code}")
    log_func(f"✔ {desc} finished.")
    return out

def download_dir(sftp, remote_dir, local_dir, log_func):
    """Recursively download a directory from remote pod."""
    try:
        entries = sftp.listdir_attr(remote_dir)
    except FileNotFoundError:
        log_func(f"⚠ No output directory found at {remote_dir}")
        return
    os.makedirs(local_dir, exist_ok=True)
    for entry in entries:
        remote_path = posixpath.join(remote_dir, entry.filename)
        local_path = os.path.join(local_dir, entry.filename)
        if stat.S_ISDIR(entry.st_mode):
            download_dir(sftp, remote_path, local_path, log_func)
        else:
            sftp.get(remote_path, local_path)
            log_func(f"  ↓ {local_path}")

In [18]:
def create_pod_with_retries(log_func, max_attempts=3, retry_delay=10, **kwargs):
    """
    Attempt to create a RunPod pod with retries.
    Returns the created pod dict.
    Raises RuntimeError after max failures.
    """
    last_err = None
    for attempt in range(1, max_attempts + 1):
        try:
            log_func(f"Attempt {attempt}/{max_attempts}: creating pod...")
            pod = runpod.create_pod(**kwargs)
            pod_id = pod.get("id")
            log_func(f"Pod created: {pod_id}")
            return pod
        except Exception as e:
            last_err = e
            log_func(f"⚠ Pod creation failed on attempt {attempt}: {e}")
            if attempt < max_attempts:
                log_func(f"Waiting {retry_delay} seconds before retrying...")
                time.sleep(retry_delay)

    raise RuntimeError(f"❌ Failed to create pod after {max_attempts} attempts. Last error: {last_err}")


# Discover Models to Judge

In [19]:
import pandas as pd
import re

def build_multi_judge_progress_matrix(
    phase4_root: Path,
    phase5_root: Path,
    task_name: str,
    judge_model: str,
    prompt_id: str
) -> pd.DataFrame:
    """Build progress matrix to identify which models need judging."""
    p4 = phase4_root.parent / task_name
    p5 = phase5_root

    if not p4.exists():
        raise FileNotFoundError(f"Phase 4 directory missing: {p4}")

    model_folders = sorted([d.name for d in p4.iterdir() if d.is_dir()])
    judge_suffix = judge_model.replace(":", "_")
    target_pattern = re.compile(rf"__{judge_suffix}-{prompt_id}\.jsonl$")

    rows = []

    for model in model_folders:
        model_p4_dir = p4 / model
        model_p5_dir = p5 / model

        # Total samples from Phase 4
        sample_files = [
            f for f in model_p4_dir.iterdir()
            if f.name.startswith(f"samples_{task_name}_") and f.suffix == ".jsonl"
        ]

        total_samples = 0
        if sample_files:
            sf = sample_files[0]
            with open(sf, "r", encoding="utf-8") as fh:
                total_samples = sum(1 for _ in fh)

        # Check for existing judgments
        judged_count = 0
        if model_p5_dir.exists():
            for jf in model_p5_dir.iterdir():
                if jf.is_file() and target_pattern.search(jf.name):
                    with open(jf, "r", encoding="utf-8") as fh:
                        judged_count = sum(1 for _ in fh)
                    break

        if total_samples == 0:
            status = "no_samples"
        elif judged_count == 0:
            status = "not_started"
        elif judged_count < total_samples:
            status = "partial"
        else:
            status = "complete"

        rows.append({
            "model_name": model,
            "ollama_name": model.replace("__", ":"),
            "judged_count": judged_count,
            "total_samples": total_samples,
            "progress_fraction": judged_count / total_samples if total_samples > 0 else 0.0,
            "status": status,
        })

    df = pd.DataFrame(rows)
    return df.sort_values("model_name")

# Build progress matrix
progress_df = build_multi_judge_progress_matrix(
    PHASE4_ROOT,
    PHASE5_ROOT,
    TASK_NAME,
    JUDGE_MODEL,
    PROMPT_ID
)

print("\n=== Judge Progress Matrix ===")
display(progress_df)

# Extract models that need judging
models_to_judge = progress_df[
    progress_df["status"].isin(["not_started", "partial"])
]["model_name"].tolist()

print(f"\nModels needing judgment: {len(models_to_judge)}")
print(models_to_judge)


=== Judge Progress Matrix ===


,model_name,ollama_name,judged_count,total_samples,progress_fraction,status
0,anthropic__claude-sonnet-4.5,anthropic:claude-sonnet-4.5,0,845,0.0,not_started
1,devstral__24b,devstral:24b,0,845,0.0,not_started
2,gemma3__12b,gemma3:12b,0,845,0.0,not_started
3,gemma3__1b,gemma3:1b,0,845,0.0,not_started
4,gemma3__27b,gemma3:27b,0,845,0.0,not_started
5,gemma3__4b,gemma3:4b,0,845,0.0,not_started
6,google__gemini-2.5-flash,google:gemini-2.5-flash,0,845,0.0,not_started
7,llama3.2__1b,llama3.2:1b,0,845,0.0,not_started
8,llama3.2__3b,llama3.2:3b,0,845,0.0,not_started
9,llama3.3__70b,llama3.3:70b,0,845,0.0,not_started



Models needing judgment: 19
['anthropic__claude-sonnet-4.5', 'devstral__24b', 'gemma3__12b', 'gemma3__1b', 'gemma3__27b', 'gemma3__4b', 'google__gemini-2.5-flash', 'llama3.2__1b', 'llama3.2__3b', 'llama3.3__70b', 'llama4__16x17b', 'mistral-large__123b', 'mistral-small3.2__24b', 'mixtral__8x22b', 'openai__gpt-4.1', 'phi3.5__3.8b', 'phi3__14b', 'phi4-mini__3.8b', 'phi4__14b']


# Pod Worker Function

This function runs on a single RunPod pod and judges all assigned models.

In [10]:
def judge_single_pod(
    assigned_models,
    judge_model,
    prompt_id,
    judge_prompt,
    image_name,
    gpu_type,
    gpu_count,
    container_disk_gb,
    phase4_root,
    phase5_root,
    task_name,
    temperature,
    max_tokens,
    sample_n,
    ssh_key_path,
    log_dir,
):
    """
    Provision a RunPod pod, judge all assigned models (locally),
    using the pod's Ollama server for inference via the RunPod proxy URL.

    Returns (success_boolean, list_of_judged_models).
    """
    pod_id = None
    ssh = None

    # Create log file
    log_file = Path(log_dir) / f"pod_{int(time.time())}.log"
    log_file.parent.mkdir(parents=True, exist_ok=True)

    def log(msg):
        stamp = time.strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{stamp}] {msg}"
        print(line)
        with open(log_file, "a", encoding="utf-8") as lf:
            lf.write(line + "\n")

    try:
        # ================================================================
        # 1. CREATE POD WITH RETRIES
        # ================================================================
        log(f"Creating pod for models: {assigned_models}")
        pod = create_pod_with_retries(
            log_func=log,
            max_attempts=3,
            retry_delay=10,
            name=f"llm-judge-{judge_model.replace(':','-')}-{int(time.time())}",
            image_name=image_name,
            gpu_type_id=gpu_type,
            gpu_count=gpu_count,
            container_disk_in_gb=container_disk_gb,
            min_vcpu_count=4,
            min_memory_in_gb=16,
            ports="22/tcp,11434/http",
            env={"OLLAMA_HOST": "0.0.0.0", "PYTHONUNBUFFERED": "1"},
            support_public_ip=True,
            start_ssh=True,
        )
        pod_id = pod["id"]
        log(f"Created pod: {pod_id}")

        # ================================================================
        # 2. WAIT FOR POD TO BE RUNNING & DISCOVER SSH PORT
        #    (Ollama is accessed via RunPod proxy URL, not via runtime ports)
        # ================================================================
        ssh_host = ssh_port = None
        max_checks = 60  # 60 * 10s = 10 minutes
        check_idx = 0

        while True:
            check_idx += 1
            details = runpod.get_pod(pod_id)
            status = details.get("desiredStatus")
            runtime = details.get("runtime")

            log(f"[RUNNING LOOP] Iteration {check_idx}")
            log(f"  desiredStatus: {status}")
            log(f"  runtime: {'None' if runtime is None else 'present'}")

            if runtime and runtime.get("ports"):
                log(f"  runtime.ports:")
                for p in runtime["ports"]:
                    log(f"    {p}")
                    # Find public SSH port
                    if (
                        p.get("type") == "tcp"
                        and p.get("privatePort") == 22
                        and p.get("isIpPublic")
                    ):
                        ssh_host = p.get("ip")
                        ssh_port = p.get("publicPort")

                log(f"  Selected ssh_host={ssh_host}, ssh_port={ssh_port}")

            if status == "RUNNING" and ssh_host:
                log("Pod is RUNNING and SSH is available.")
                break

            if check_idx >= max_checks:
                raise RuntimeError("Timed out waiting for pod SSH to become ready.")

            time.sleep(10)

        # *** IMPORTANT CHANGE: use RunPod proxy URL for Ollama ***
        proxy_base = f"https://{pod_id}-11434.proxy.runpod.net"
        judge_url = f"{proxy_base}/v1/chat/completions"
        log(f"Using judge_url (RunPod proxy) = {judge_url}")

        # ================================================================
        # 3. CONNECT VIA SSH (for provisioning only)
        # ================================================================
        ssh = paramiko.SSHClient()
        ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        ssh.connect(
            ssh_host,
            port=ssh_port,
            username="root",
            pkey=paramiko.Ed25519Key.from_private_key_file(ssh_key_path),
        )
        log("SSH connected.")

        # ================================================================
        # 4. PROVISION ENVIRONMENT ON POD
        # ================================================================
        provisioning_steps = [
            ("Install lshw", "apt update && apt install -y lshw"),
            ("Install Ollama", "curl -fsSL https://ollama.com/install.sh | sh"),
            (
                "Start Ollama",
                "nohup env OLLAMA_HOST=0.0.0.0 ollama serve > /tmp/ollama.log 2>&1 &",
            ),
            ("Wait for Ollama", "sleep 10"),
            (f"Pull judge model {judge_model}", f"ollama pull {judge_model}"),
        ]
        for desc, cmd in provisioning_steps:
            run_and_check(ssh, cmd, desc, log)

        log("Pod provisioned successfully.")

        # ================================================================
        # 5. JUDGE EACH ASSIGNED MODEL
        # ================================================================
        judged_models = []

        for model_name in assigned_models:
            log(f"\n{'='*60}")
            log(f"Starting judgment for model: {model_name}")
            log(f"{'='*60}")

            try:
                # Load Phase 4 samples
                model_p4_dir = phase4_root.parent / task_name / model_name
                src_samples = newest(model_p4_dir.glob("samples_*.jsonl"))
                if not src_samples:
                    log(f"[skip] {model_name}: no Phase 4 samples found")
                    continue

                source_rows = load_jsonl(src_samples)
                if sample_n > 0:
                    source_rows = source_rows[:sample_n]
                total = len(source_rows)
                log(f"Loaded {total} samples from {src_samples.name}")

                # Prepare Phase 5 output
                out_dir = phase5_root / model_name
                out_dir.mkdir(parents=True, exist_ok=True)

                phase4_name = src_samples.name.replace(".jsonl", "")
                judge_suffix = judge_model.replace(":", "_")
                if prompt_id:
                    filename = f"{phase4_name}__{judge_suffix}-{prompt_id}.jsonl"
                else:
                    filename = f"{phase4_name}__{judge_suffix}.jsonl"

                final_path = out_dir / filename

                # Check resume status
                if not final_path.exists():
                    with open(final_path, "w", encoding="utf-8"):
                        pass
                    log(f"[new] Created {filename}")
                else:
                    log(f"[resume] Continuing {filename}")

                # Load existing judgments
                done_ids = set()
                existing_by_id = {}
                for row in load_jsonl(final_path):
                    sid = row.get("sample_id")
                    if isinstance(sid, int):
                        done_ids.add(sid)
                        existing_by_id[sid] = row

                # Alignment checks
                for sid in done_ids:
                    if sid >= total:
                        raise RuntimeError(
                            f"{filename} has sample_id {sid} beyond source length {total}"
                        )
                    ensure_alignment_or_die(existing_by_id[sid], source_rows[sid], sid)

                log(f"Progress: {len(done_ids)}/{total} judged")

                # Determine what needs judging
                to_do = [i for i in range(total) if i not in done_ids]
                if not to_do:
                    log(f"[complete] {model_name}: nothing to judge")
                    judged_models.append(model_name)
                    continue

                # ============================================================
                # JUDGE SAMPLES SEQUENTIALLY (INFERENCE HAPPENS ON POD VIA PROXY)
                # ============================================================
                log(f"Judging {len(to_do)} samples...")

                with open(final_path, "a", encoding="utf-8") as fout:
                    for sid in tqdm(to_do, desc=f"Judging {model_name}", unit="sample"):
                        phase4_row = deepcopy(source_rows[sid])
                        fields_for_prompt = derive_prompt_fields(phase4_row)

                        # Check for missing fields
                        missing = triad_missing(fields_for_prompt)
                        if missing:
                            record = {
                                "sample_id": sid,
                                "phase4_row": phase4_row,
                                "judge": {
                                    "fields": None,
                                    "prompt": None,
                                    "raw_output": None,
                                    "error": f"missing_fields: {missing}",
                                    "timestamp": datetime.now().isoformat(),
                                    "meta": {
                                        "judge_model": judge_model,
                                        "temperature": temperature,
                                        "max_tokens": max_tokens,
                                        "task_name": task_name,
                                        "model_name": model_name,
                                        "prompt_id": prompt_id,
                                    },
                                },
                            }
                            fout.write(json.dumps(record) + "\n")
                            fout.flush()
                            continue

                        # Build judge prompt
                        prompt = judge_prompt.format(**fields_for_prompt)

                        raw = None
                        parsed = None
                        error = None

                        try:
                            response = requests.post(
                                judge_url,
                                headers={"Content-Type": "application/json"},
                                json={
                                    "model": judge_model,
                                    "messages": [
                                        {
                                            "role": "system",
                                            "content": (
                                                "You are an expert evaluator in "
                                                "systems engineering education."
                                            ),
                                        },
                                        {"role": "user", "content": prompt},
                                    ],
                                    "temperature": temperature,
                                    "max_tokens": max_tokens,
                                },
                                timeout=120,
                            )
                            data = response.json()
                            raw = data["choices"][0]["message"]["content"]
                            parsed = safe_json(raw)
                        except Exception as e:
                            error = str(e)

                        # Parse response based on prompt type
                        if prompt_id == "p1":
                            fields = {
                                "technical_accuracy": {"score": None},
                                "conceptual_understanding": {"score": None},
                                "completeness": {"score": None},
                                "clarity_organization": {"score": None},
                                "professional_relevance": {"score": None},
                                "overall_score": None,
                            }
                            if parsed:
                                fields["technical_accuracy"]["score"] = (
                                    (parsed.get("technical_accuracy") or {}).get("score")
                                )
                                fields["conceptual_understanding"]["score"] = (
                                    (parsed.get("conceptual_understanding") or {}).get(
                                        "score"
                                    )
                                )
                                fields["completeness"]["score"] = (
                                    (parsed.get("completeness") or {}).get("score")
                                )
                                fields["clarity_organization"]["score"] = (
                                    (parsed.get("clarity_organization") or {}).get(
                                        "score"
                                    )
                                )
                                fields["professional_relevance"]["score"] = (
                                    (parsed.get("professional_relevance") or {}).get(
                                        "score"
                                    )
                                )
                                fields["overall_score"] = parsed.get("overall_score")
                        else:  # p2
                            fields = {
                                "technical_accuracy": {
                                    "score": None,
                                    "justification": None,
                                },
                                "conceptual_understanding": {
                                    "score": None,
                                    "justification": None,
                                },
                                "completeness": {
                                    "score": None,
                                    "justification": None,
                                },
                                "clarity_organization": {
                                    "score": None,
                                    "justification": None,
                                },
                                "professional_relevance": {
                                    "score": None,
                                    "justification": None,
                                },
                                "overall_score": None,
                                "overall_assessment": None,
                                "key_strengths": None,
                                "improvement_areas": None,
                            }
                            if parsed:
                                for key in [
                                    "technical_accuracy",
                                    "conceptual_understanding",
                                    "completeness",
                                    "clarity_organization",
                                    "professional_relevance",
                                ]:
                                    fields[key] = {
                                        "score": (parsed.get(key) or {}).get("score"),
                                        "justification": (
                                            parsed.get(key) or {}
                                        ).get("justification"),
                                    }
                                fields["overall_score"] = parsed.get("overall_score")
                                fields["overall_assessment"] = parsed.get(
                                    "overall_assessment"
                                )
                                fields["key_strengths"] = parsed.get("key_strengths")
                                fields["improvement_areas"] = parsed.get(
                                    "improvement_areas"
                                )

                        # Write record
                        record = {
                            "sample_id": sid,
                            "phase4_row": phase4_row,
                            "judge": {
                                "fields": fields,
                                "prompt": prompt,
                                "raw_output": raw,
                                "error": error,
                                "timestamp": datetime.now().isoformat(),
                                "meta": {
                                    "judge_model": judge_model,
                                    "temperature": temperature,
                                    "max_tokens": max_tokens,
                                    "task_name": task_name,
                                    "model_name": model_name,
                                    "prompt_id": prompt_id,
                                },
                            },
                        }
                        fout.write(json.dumps(record) + "\n")
                        fout.flush()

                log(f"[✓] Completed judging {model_name}")
                judged_models.append(model_name)

            except Exception as e:
                log(f"[✗] Error judging {model_name}: {e}")
                continue

        log(f"\nPod completed. Judged {len(judged_models)} models: {judged_models}")
        return True, judged_models

    except Exception as e:
        log(f"[✗] Pod failed: {e}")
        return False, []

    finally:
        # Cleanup
        if ssh:
            ssh.close()
        if pod_id:
            try:
                runpod.terminate_pod(pod_id)
                log(f"Pod {pod_id} terminated.")
            except Exception as e:
                log(f"⚠ Pod termination failed: {e}")

# Main Execution: Parallel Pod Distribution

In [11]:
def run_llm_judge_parallel(
    models_to_judge,
    max_concurrent_pods,
    judge_model,
    prompt_id,
    judge_prompt,
    image_name,
    gpu_type,
    gpu_count,
    container_disk_gb,
    phase4_root,
    phase5_root,
    task_name,
    temperature,
    max_tokens,
    sample_n,
    ssh_key_path,
    log_dir,
):
    """
    Distribute models across multiple RunPod pods and judge in parallel.
    """
    if not models_to_judge:
        print("No models to judge!")
        return

    print(f"\n{'='*60}")
    print(f"Starting parallel LLM judge with {max_concurrent_pods} pods")
    print(f"Judge model: {judge_model}")
    print(f"Prompt ID: {prompt_id}")
    print(f"Models to judge: {len(models_to_judge)}")
    print(f"{'='*60}\n")

    # Distribute models across pods
    models_per_pod = len(models_to_judge) // max_concurrent_pods
    remainder = len(models_to_judge) % max_concurrent_pods
    
    pod_assignments = []
    start_idx = 0
    for i in range(max_concurrent_pods):
        # Give extra model to first 'remainder' pods
        count = models_per_pod + (1 if i < remainder else 0)
        end_idx = start_idx + count
        pod_assignments.append(models_to_judge[start_idx:end_idx])
        start_idx = end_idx

    print("Pod assignments:")
    for i, assignment in enumerate(pod_assignments):
        print(f"  Pod {i}: {len(assignment)} models - {assignment}")
    print()

    # Run pods in parallel
    results = {}
    with ThreadPoolExecutor(max_workers=max_concurrent_pods) as executor:
        futures = {}
        for i, assigned_models in enumerate(pod_assignments):
            if not assigned_models:
                continue
            future = executor.submit(
                judge_single_pod,
                assigned_models,
                judge_model,
                prompt_id,
                judge_prompt,
                image_name,
                gpu_type,
                gpu_count,
                container_disk_gb,
                phase4_root,
                phase5_root,
                task_name,
                temperature,
                max_tokens,
                sample_n,
                ssh_key_path,
                log_dir,
            )
            futures[future] = i

        for future in as_completed(futures):
            pod_idx = futures[future]
            success, judged_models = future.result()
            results[pod_idx] = {"success": success, "judged_models": judged_models}
            print(f"\n[SUMMARY] Pod {pod_idx} finished: {'✅ success' if success else '❌ failed'}")
            print(f"  Judged models: {judged_models}")

    print(f"\n{'='*60}")
    print("All pods completed!")
    print(f"{'='*60}\n")
    
    # Summary
    total_judged = sum(len(r["judged_models"]) for r in results.values())
    print(f"Total models judged: {total_judged}/{len(models_to_judge)}")
    
    return results

# Run Judgment

In [12]:
if models_to_judge:
    results = run_llm_judge_parallel(
        models_to_judge=models_to_judge,
        max_concurrent_pods=MAX_CONCURRENT_PODS,
        judge_model=JUDGE_MODEL,
        prompt_id=PROMPT_ID,
        judge_prompt=JUDGE_PROMPT,
        image_name=IMAGE_NAME,
        gpu_type=GPU_TYPE,
        gpu_count=GPU_COUNT,
        container_disk_gb=CONTAINER_DISK_GB,
        phase4_root=PHASE4_ROOT,
        phase5_root=PHASE5_ROOT,
        task_name=TASK_NAME,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        sample_n=SAMPLE_N,
        ssh_key_path=SSH_KEY_PATH,
        log_dir=LOG_DIR,
    )
else:
    print("No models need judging!")


Starting parallel LLM judge with 2 pods
Judge model: gpt-oss:120b
Prompt ID: p1
Models to judge: 19

Pod assignments:
  Pod 0: 10 models - ['anthropic__claude-sonnet-4.5', 'devstral__24b', 'gemma3__12b', 'gemma3__1b', 'gemma3__27b', 'gemma3__4b', 'google__gemini-2.5-flash', 'llama3.2__1b', 'llama3.2__3b', 'llama3.3__70b']
  Pod 1: 9 models - ['llama4__16x17b', 'mistral-large__123b', 'mistral-small3.2__24b', 'mixtral__8x22b', 'openai__gpt-4.1', 'phi3.5__3.8b', 'phi3__14b', 'phi4-mini__3.8b', 'phi4__14b']

[2025-11-25 01:20:13] Creating pod for models: ['anthropic__claude-sonnet-4.5', 'devstral__24b', 'gemma3__12b', 'gemma3__1b', 'gemma3__27b', 'gemma3__4b', 'google__gemini-2.5-flash', 'llama3.2__1b', 'llama3.2__3b', 'llama3.3__70b']
[2025-11-25 01:20:13] Creating pod for models: ['llama4__16x17b', 'mistral-large__123b', 'mistral-small3.2__24b', 'mixtral__8x22b', 'openai__gpt-4.1', 'phi3.5__3.8b', 'phi3__14b', 'phi4-mini__3.8b', 'phi4__14b']
[2025-11-25 01:20:13] Attempt 1/3: creating p

Judging llama4__16x17b:   0%|          | 0/5 [00:00<?, ?sample/s]

[2025-11-25 01:24:54] stderr: pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling 6be6d66a3f54:   0% ▕                  ▏ 267 KB/ 65 GB                  pulling manifest 
pulling 6be6d66a3f54:   0% ▕                  ▏  14 MB/ 65 GB                  pulling manifest 
pulling 6be6d66a3f54:   0% ▕                  ▏  53 MB/ 65 GB                  pulling manifest 
pulling 6be6d66a3f54:   0% ▕                  ▏ 126 MB/ 65 GB                  pulling manifest 
pulling 6be6d66a3f54:   0% ▕                  ▏ 164 MB/ 65 GB                  pulling manifest 
pulling 6be6d66a3f54:   0% ▕                  ▏ 248 MB/ 65 GB                  pulling manifest 
pulling 6be6d66a3f54:   0% ▕                  ▏ 326 MB/ 65 GB                  pulling manifest 
pulling 6be6d66a3f54:   1% ▕                  ▏ 365 MB/ 65 GB                  pulling manifest 
pulling 6be6d66a3f54:   1% ▕                  ▏ 443 MB/ 65 GB                  pu

Judging anthropic__claude-sonnet-4.5:   0%|          | 0/5 [00:00<?, ?sample/s]

[2025-11-25 01:25:10] [✓] Completed judging llama4__16x17b
[2025-11-25 01:25:10] 
[2025-11-25 01:25:10] Starting judgment for model: mistral-large__123b
[2025-11-25 01:25:10] ============================================================
[2025-11-25 01:25:10] Loaded 5 samples from samples_sysengbench-osq_2025-11-15T00-00-12.395367.jsonl
[2025-11-25 01:25:10] [new] Created samples_sysengbench-osq_2025-11-15T00-00-12.395367__gpt-oss_120b-p1.jsonl
[2025-11-25 01:25:10] Progress: 0/5 judged
[2025-11-25 01:25:10] Judging 5 samples...


...

[2025-11-25 01:27:20] [✓] Completed judging llama3.2__1b
[2025-11-25 01:27:20] 
[2025-11-25 01:27:20] Starting judgment for model: llama3.2__3b
[2025-11-25 01:27:20] ============================================================
[2025-11-25 01:27:20] Loaded 5 samples from samples_sysengbench-osq_2025-11-19T21-47-50.052256.jsonl
[2025-11-25 01:27:20] [new] Created samples_sysengbench-osq_2025-11-19T21-47-50.052256__gpt-oss_120b-p1.jsonl
[2025-11-25 01:27:20] Progress: 0/5 judged
[2025-11-25 01:27:20] Judging 5 samples...


Judging llama3.2__3b:   0%|          | 0/5 [00:00<?, ?sample/s]

[2025-11-25 01:27:29] [✓] Completed judging phi4__14b
[2025-11-25 01:27:29] 
Pod completed. Judged 9 models: ['llama4__16x17b', 'mistral-large__123b', 'mistral-small3.2__24b', 'mixtral__8x22b', 'openai__gpt-4.1', 'phi3.5__3.8b', 'phi3__14b', 'phi4-mini__3.8b', 'phi4__14b']
[2025-11-25 01:27:30] Pod 8rg6xrooerfdfs terminated.

[SUMMARY] Pod 1 finished: ✅ success
  Judged models: ['llama4__16x17b', 'mistral-large__123b', 'mistral-small3.2__24b', 'mixtral__8x22b', 'openai__gpt-4.1', 'phi3.5__3.8b', 'phi3__14b', 'phi4-mini__3.8b', 'phi4__14b']
[2025-11-25 01:27:32] [✓] Completed judging llama3.2__3b
[2025-11-25 01:27:32] 
[2025-11-25 01:27:32] Starting judgment for model: llama3.3__70b
[2025-11-25 01:27:32] ============================================================
[2025-11-25 01:27:32] Loaded 5 samples from samples_sysengbench-osq_2025-11-14T23-24-28.915214.jsonl
[2025-11-25 01:27:32] [new] Created samples_sysengbench-osq_2025-11-14T23-24-28.915214__gpt-oss_120b-p1.jsonl
[2025-11-25 01:

Judging llama3.3__70b:   0%|          | 0/5 [00:00<?, ?sample/s]

[2025-11-25 01:27:47] [✓] Completed judging llama3.3__70b
[2025-11-25 01:27:47] 
Pod completed. Judged 10 models: ['anthropic__claude-sonnet-4.5', 'devstral__24b', 'gemma3__12b', 'gemma3__1b', 'gemma3__27b', 'gemma3__4b', 'google__gemini-2.5-flash', 'llama3.2__1b', 'llama3.2__3b', 'llama3.3__70b']
[2025-11-25 01:27:47] Pod a8uvrosy8n6lce terminated.

[SUMMARY] Pod 0 finished: ✅ success
  Judged models: ['anthropic__claude-sonnet-4.5', 'devstral__24b', 'gemma3__12b', 'gemma3__1b', 'gemma3__27b', 'gemma3__4b', 'google__gemini-2.5-flash', 'llama3.2__1b', 'llama3.2__3b', 'llama3.3__70b']

All pods completed!

Total models judged: 19/19


# Verify Results

In [ ]:
# Rebuild progress matrix to see updated status
final_progress_df = build_multi_judge_progress_matrix(
    PHASE4_ROOT,
    PHASE5_ROOT,
    TASK_NAME,
    JUDGE_MODEL,
    PROMPT_ID
)

print("\n=== Final Judge Progress Matrix ===")
display(final_progress_df)

incomplete = final_progress_df[
    final_progress_df["status"].isin(["not_started", "partial"])
]
print(f"\nRemaining models: {len(incomplete)}")
if len(incomplete) > 0:
    display(incomplete)